# OPERADORES

## 1. Clásicos
```bash
+   suma
-   resta
*   multiplicación
/   división
%   módulo

```

### Forma recomendada `(( ))`
Es la forma más moderna y limpia 

In [2]:
%%bash
(( resultado = 5 + 3 ))
echo "$resultado"

8


### Usarlo con `$(( ))` (expansión aritmética)
Esto es muy común cuando quieres asignar:

In [6]:
%%bash
#/bin/bash

resultado=$(( 5 + 3))
echo "$resultado"

8


In [9]:
%%bash
#!/bin/bash

# También con variables
a=10
b=5

resultado=$(( a * b)) # No es necesario usar $
echo "$resultado"


50


In [4]:
%%bash
### Ejemplo completo:

echo $((10 + 5))   # 15
echo $((10 - 5))   # 5
echo $((10 * 5))   # 50
echo $((10 / 5))   # 2
echo $((10 % 3))   # 1

15
5
50
2
1


Bash si tiene aritmética, pero:
  * Solo trabaja con enteros (integers)
  * No soporta decimales (floats)


Ejemplo:
```bash
(( a = 5/3 ))
echo $a
```
Salida:
```ouput
1
```

> Bash trunca, no redondea.

Con decimales:
```bash
(( a=5 / 2.0 ))
```
Salida 
```ouput
bash: ((: 5 / 2.0: syntax error: invalid arithmetic operator
```

> Bash no entiende números con punto decimal



#### Como manejar decimales
Aquí entra un comando `bc`

`bc` es una calculadora externa (un programa aparte), no es parte del núcleo de Bash.

Sirve para:
  * Decimales
  * Precisión arbitraria
  * Funciones matemáticas (con `-l`)
  * Expresiones más complejas

In [16]:
%%bash
# Uso básico
echo "2 + 3" | bc
echo "5 / 3" | bc


5
1


Usado con librerías matemáticas (`-l`)

In [ ]:
%%bash
echo " 5 / 3" | bc -l|

1.66666666666666666666


Usado con scale

In [21]:
%%bash
# Solo dos decimales
echo "scale=2; 5 / 3 " | bc

1.66


Idea:
  * `(( ))` → ejecutar operaciones
  * `$(( ))` → obtener resultado

## 2. Comparación

### Números
```bash
-eq  # igual
-ne  # distinto
-lt  # menor
-le  # menor o igual
-gt  # mayor
-ge  # mayor o igual
```


Se usa junto con `[ ]` 

```bash
[ condicion ]
```
Es en realidad un  comando (alias de `test`), no sintaxis mágica.

  * Devuelve 0 -> `verdadero`
  * Devuelve 1 -> `falso`

> Dejar espacios al borde de los paréntisis `[ condición ]`

> Verificar con `$?` que nos muestra el código de salida del último comando


In [ ]:
%%bash

[ 4 -eq 4 ] # Es igual
echo "Devuelve: $?" # Verdadero = 0

[ 4 -ne 4 ] # No es igual
echo "Devuelve: $?" # Falso = 1

[ 4 -lt 3 ] #  Es menor
echo "Devuelve: $?"

[ 4 -gt 6 ] # Es mayor
echo "Devuelve: $?"


Devuelve: 0
Devuelve: 1
Devuelve: 1
Devuelve: 1


### Strings
```bash
=    # igual
!=   # distinto
-z   # string vacío
-n   # string no vacío
```

También se usa con `[ ]` 


In [57]:
%%bash
var1="Hola"
var2="hola"
var3=""

[ "Si" = "si" ]
echo $?

[ "$var1" != "$var2" ]
echo $?

[ -z "$var3" ]
echo $?

[ -n "$var3" ]
echo $?



1
0
0
1


### Lógicos
```bash
&&  # AND : Ambas condiciones deben ser verdaderas
||  # OR  : Al menos uno deben ser verdaderas
!   # NOT : Niega la condición
```

> Precedencia: ! > && > ||

#### Limitaciones de `[ ]`
No soporta bien:
  * Expresiones complejas
  * `&&` y `||` directamente dentro
  * Comparaciones modernas
  * Problemas con espacios si no usas comillas

Ejemplo problemático:

In [61]:
%%bash
a=""
[ $a = "Hola" ]
echo $?

bash: line 2: [: =: unary operator expected


2


In [63]:
%%bash
a=""
# Con comillas
[ "$a" = "Hola" ]
echo $?

1


#### Mejor opción `[[ ]]`
```bash
[[ condición ]]
```
> Es una estructura moderna de Bash.

Ventajas:
  * No necesita tantas comillas
  * Permite `&&` y `||`
  * Mejor para strings

In [68]:
%%bash
a="Hola"
b="mundo"

[[ $a == hola && $b == mundo ]]
echo $?

1


Diferencias entre `=` y `==`

Funcionan igual
```bash
[[ $a = hola ]]
[[ $a == hola ]]
```

Pero `==` permite patrones
```bash
[[ $a == h* ]]
```

In [70]:
%%bash
a="hola"

[[ $a == h* ]] && echo "match"

match


### Archivos
Se usan dentro de [ ] o [[ ]] para verificar el estado de archivos o rutas.

#### `-f` 
Archivo normal

In [ ]:
%%bash

[ -f 0-shell.ipynb ] 
echo $? # Existe

# Combinando
[ -f 0-shell.ipynb ] && echo "Es un archivo" # && Si el primer comando falla el siguiente no se ejecuta

0
Es un archivo


#### `-d`
Es un directorio

In [78]:
%%bash
[ -f carpeta ]
echo $?

1


#### `-e`
Si existe ya sea archivo o carpeta

In [79]:
%%bash

[[ -e archivo.txt ]]
echo $?

1


#### `-w`
Se puede escribir

In [81]:
%%bash
[ -w 0-shell.ipynb ]
echo $?

0


#### `-x` 
Es ejecutable

In [82]:
%%bash
[[ -x 0-shell.ipynb ]]
echo $?

1


#### `-s` 
No esta vacío

In [83]:
%%bash
[[ -x 0-shell.ipynb ]]
echo $?

1


#### `-L`
Es enlace simbólico

In [84]:
%%bash
[ -L 0-shell.ipynb ] 
echo $?

1


#### `-nt`
Comparación entre archivos -> `más nuevo que`

In [89]:
%%bash
[ 2-ejecucion.ipynb -nt 0-shell.ipynb ] 
echo $?

0


In [96]:
ls -l 2-ejecucion.ipynb 0-shell.ipynb | cut -d" " -f"6-9"

Mar 18 06:04 0-shell.ipynb
Mar 19 00:41 2-ejecucion.ipynb


#### `-ot`
Comparación entre archivos -> `más viejo que`

In [101]:
%%bash

[ 0-shell.ipynb -ot 2-ejecucion.ipynb ] 
echo $?

0


#### Problema clásico
Suponer que hay un archivo con espacios:
```bash
archivo="mi archivo.txt"
```

Usando `[ ]` (forma incorrecta)
```bash
[ -f $archivo ]
```
Bash lo interpreta como:
```bash
[ -f mi archivo.txt ]
```
Error:
  * Cree que `mi` y `archivo.txt` son argumentos separados
  * Resultado: falla o comportamiento raro

***Solución con `[ ]`*** (correcta)
```bash
[ -f "$archivo" ] 
```
> Las comillas evitan que se rompa

Con `[[ ]]` (más seguro)
```bash
[[ -f $archivo ]] 
```
  * Funciona sin comillas
  * No rompe por espacios


#### Regla de oro
Si usas `[ ]`
```bash
SIEMPRE: "$variable"
```
Si usas `[[ ]]`
  * Más tolerante
  * Pero igual usar comillas es buena práctica
